# NGB v4 architecture comparison

Compares the one-block/one-head control with the distinct four-block/four-head
small language model using the same matched seeds and optimizer recipes. The
paired tables use seed-matched architecture differences.


In [ ]:
ONE_HEAD_CONFIG = "configs/v4_one_head.yaml"
SMALL_4X4_CONFIG = "configs/v4_small_4x4.yaml"
SEEDS = ""
NGB_STORAGE_ROOT = "/tmp/rg-ngb"


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / "baseline" / "ngb"]
NGB_ROOT_DIR = next(
    (path for path in candidates if (path / "configs" / "v4_one_head.yaml").is_file()),
    None,
)
if NGB_ROOT_DIR is None:
    raise FileNotFoundError("Run from baseline/ngb or the repository root")
RUNTIME_SRC = NGB_ROOT_DIR.parent / "nanogpt_one_head" / "src"
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))

import math
import matplotlib.pyplot as plt
import numpy as np

from rg_nanogpt_one_head import (
    OPTIMIZER_COLORS,
    OPTIMIZER_LABELS,
    SUPPORTED_OPTIMIZERS,
    discover_matched_complete_seeds,
    final_test_summary,
    load_config,
    load_epoch_metrics,
    load_spectral_summary,
    load_test_results,
    mean_ci95,
    run_slug,
)
from rg_nanogpt_one_head.analysis import summarize_by_epoch
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


In [ ]:
configs = {
    "one_head": load_config(NGB_ROOT_DIR / ONE_HEAD_CONFIG),
    "small_4x4": load_config(NGB_ROOT_DIR / SMALL_4X4_CONFIG),
}
optimizers = tuple(SUPPORTED_OPTIMIZERS)
results_roots = {
    architecture: Path(NGB_STORAGE_ROOT) / "results" / run_slug(cfg)
    for architecture, cfg in configs.items()
}
available = {
    architecture: set(discover_matched_complete_seeds(root, optimizers=optimizers))
    for architecture, root in results_roots.items()
}
seeds = (
    tuple(int(value.strip()) for value in SEEDS.split(",") if value.strip())
    if SEEDS.strip()
    else tuple(sorted(set.intersection(*available.values())))
)
if not seeds:
    raise RuntimeError(f"No complete seed intersection across architectures: {available}")
print("matched architecture seeds:", seeds)
print("result roots:", results_roots)

all_epoch = []
all_spectral = []
all_test = []
for architecture, root in results_roots.items():
    epoch = load_epoch_metrics(root, optimizers=optimizers, seeds=seeds)
    spectral = load_spectral_summary(root, optimizers=optimizers, seeds=seeds)
    test = load_test_results(root, optimizers=optimizers, seeds=seeds)
    for frame in (epoch, spectral, test):
        frame.insert(0, "architecture", architecture)
    all_epoch.append(epoch)
    all_spectral.append(spectral)
    all_test.append(test)
epoch_metrics = pd.concat(all_epoch, ignore_index=True)
spectral_summary = pd.concat(all_spectral, ignore_index=True)
test_results = pd.concat(all_test, ignore_index=True)
plot_root = Path(NGB_STORAGE_ROOT) / "plots" / "v4_architecture_comparison"
plot_root.mkdir(parents=True, exist_ok=True)


In [ ]:
summary_frames = []
for architecture, frame in test_results.groupby("architecture"):
    part = final_test_summary(frame)
    part.insert(0, "architecture", architecture)
    summary_frames.append(part)
architecture_summary = pd.concat(summary_frames, ignore_index=True)
architecture_summary.to_csv(plot_root / "architecture_summary_95ci.csv", index=False)
display(architecture_summary.sort_values(["checkpoint", "metric", "optimizer", "architecture"]))

rows = []
for optimizer in optimizers:
    for checkpoint in ("final", "validation_selected"):
        selected = test_results[
            (test_results["optimizer"] == optimizer)
            & (test_results["checkpoint"] == checkpoint)
        ]
        for metric in ("test_loss", "test_accuracy", "test_bleu"):
            left = selected[selected["architecture"] == "small_4x4"][["seed", metric]].rename(columns={metric: "small_4x4"})
            right = selected[selected["architecture"] == "one_head"][["seed", metric]].rename(columns={metric: "one_head"})
            paired = left.merge(right, on="seed", validate="one_to_one")
            stats = mean_ci95(paired["small_4x4"] - paired["one_head"])
            rows.append({
                "optimizer": optimizer,
                "optimizer_label": OPTIMIZER_LABELS[optimizer],
                "checkpoint": checkpoint,
                "metric": metric,
                "contrast": "small_4x4 - one_head",
                **stats,
            })
architecture_paired = pd.DataFrame(rows)
architecture_paired.to_csv(plot_root / "paired_architecture_differences_95ci.csv", index=False)
display(architecture_paired.sort_values(["checkpoint", "metric", "optimizer"]))


In [ ]:
styles = {"one_head": ":", "small_4x4": "-"}
for optimizer in optimizers:
    for metric in ("val_loss", "val_accuracy", "test_loss", "test_accuracy"):
        figure, axis = plt.subplots(figsize=(9, 5))
        for architecture in ("one_head", "small_4x4"):
            subset = epoch_metrics[
                (epoch_metrics["architecture"] == architecture)
                & (epoch_metrics["optimizer"] == optimizer)
            ]
            summary = summarize_by_epoch(subset, metric, x="nominal_epoch", group=("architecture", "optimizer"))
            axis.plot(
                summary["nominal_epoch"], summary["mean"],
                color=OPTIMIZER_COLORS[optimizer], linestyle=styles[architecture],
                linewidth=2.2, label=architecture,
            )
            axis.fill_between(
                summary["nominal_epoch"], summary["ci95_lower"], summary["ci95_upper"],
                color=OPTIMIZER_COLORS[optimizer], alpha=0.10,
            )
        axis.set(xlabel="Corpus-equivalent epoch", ylabel=metric, title=f"{OPTIMIZER_LABELS[optimizer]}: one-head vs small 4x4")
        axis.grid(alpha=0.25)
        axis.legend(frameon=False)
        figure.tight_layout()
        figure.savefig(plot_root / f"{optimizer}_{metric}.png", dpi=170, bbox_inches="tight")
        plt.show()

for optimizer in optimizers:
    figure, axis = plt.subplots(figsize=(9, 5))
    for architecture in ("one_head", "small_4x4"):
        subset = spectral_summary[
            (spectral_summary["architecture"] == architecture)
            & (spectral_summary["optimizer"] == optimizer)
        ]
        summary = summarize_by_epoch(subset, "alpha_median", x="epoch", group=("architecture", "optimizer"))
        axis.plot(summary["epoch"], summary["mean"], color=OPTIMIZER_COLORS[optimizer], linestyle=styles[architecture], linewidth=2.2, label=architecture)
        axis.fill_between(summary["epoch"], summary["ci95_lower"], summary["ci95_upper"], color=OPTIMIZER_COLORS[optimizer], alpha=0.10)
    axis.axhline(2.0, color="black", linestyle="--", linewidth=1.0, label="alpha = 2")
    axis.set(xlabel="Corpus-equivalent epoch", ylabel="Median alpha", title=f"{OPTIMIZER_LABELS[optimizer]}: architecture spectral comparison")
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
    figure.tight_layout()
    figure.savefig(plot_root / f"{optimizer}_alpha_median.png", dpi=170, bbox_inches="tight")
    plt.show()
